# 🏗️ FairRecovery++ — GRPO Training Notebook**Train an LLM to make fair disaster recovery decisions using Reinforcement Learning.**| Component | Value ||---|---|| **Model** | Llama-3.2-1B-Instruct (4-bit, Unsloth) || **Algorithm** | GRPO (TRL) || **Environment** | FairRecovery++ (in-process, no network) || **Goal** | Prioritise vulnerable populations over easy-to-fix wealthy zones || **Runtime** | ~10 min on free Colab T4 |> **Themes**: Multi-Agent Interactions (#1) · Long-Horizon Planning (#2) · World Modeling (#3)

In [ ]:
# ── Cell 1: Install dependencies (restart runtime after first run) ───────────!pip install -q unsloth requests matplotlib pandas numpy structlog pydantic!pip install -q 'trl>=0.12.0' 'transformers>=4.46.0' 'accelerate>=0.34.0'!pip install -q 'peft>=0.13.0' 'bitsandbytes>=0.44.0' 'datasets>=2.0.0'print('✅ Dependencies installed. Restart runtime if this is the first run.')

In [ ]:
# ── Cell 2: Clone repo & configure ────────────────────────────────────────────import os, sys, shutilREPO_URL = 'https://github.com/joshua400/FairRecovery-PlusPlus.git'REPO_DIR = '/content/FairRecovery-PlusPlus'if os.path.exists(REPO_DIR):    shutil.rmtree(REPO_DIR)!git clone {REPO_URL} {REPO_DIR}sys.path.insert(0, REPO_DIR)os.chdir(REPO_DIR)MODEL_NAME = 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'DIFFICULTY = 'hard'NUM_EPISODES = 5MAX_STEPS_PER_EP = 20OUTPUT_DIR = './fairrecovery_grpo'print(f'Model: {MODEL_NAME}')print(f'Difficulty: {DIFFICULTY}')print(f'Repo cloned to: {REPO_DIR}')

In [ ]:
# ── Cell 3: Verify environment works (in-process, no network needed) ──────────from server.fairrecovery_environment import FairRecoveryEnvironmentfrom fairrecovery_env.models import FairRecoveryAction, AllocationItemenv = FairRecoveryEnvironment()obs = env.reset(difficulty='hard')print(f'✅ Reset OK: day={obs.day}, budget={obs.budget_left}, zones={len(obs.zones)}, stage={obs.step_stage}')# Quick step testaction = FairRecoveryAction(action_type='analyze', critical_zones=[4, 3], reasoning='test')obs = env.step(action)print(f'✅ Step OK: reward={obs.reward:.4f}, r_exec={obs.r_exec:.4f}, r_fair={obs.r_fair:.4f}')print(f'✅ Multi-agent events: {len(obs.agent_events)}')print(f'✅ Predictions available: {obs.predictions is not None}')print('\n🎯 Environment fully operational. Ready for training.')

In [ ]:
# ── Cell 4: Load model with Unsloth ──────────────────────────────────────────from unsloth import FastLanguageModelimport torchmodel, tokenizer = FastLanguageModel.from_pretrained(    model_name=MODEL_NAME,    max_seq_length=512,    load_in_4bit=True,)model = FastLanguageModel.get_peft_model(    model,    r=16,    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],    lora_alpha=16,    lora_dropout=0.0,    bias='none',    use_gradient_checkpointing='unsloth',    random_state=42,)print(f'✅ Model loaded: {MODEL_NAME}')print(f'   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ── Cell 5: Define prompts and action parser ─────────────────────────────────import json, refrom datasets import DatasetSYSTEM_PROMPT = """You are a disaster recovery coordinator allocating limited resources across damaged zones.You must balance EFFICIENCY (restore services) and FAIRNESS (prioritise vulnerable populations).Protocol (follow in order each day):1. analyze: identify the 1-2 most critical zones (high damage × high vulnerability)2. allocate: assign resources {power: cost 10, water: cost 15, medical: cost 20} to zones3. execute: commit allocations4. After all days: submitIMPORTANT: Zone 4 (high vulnerability, severe damage) matters MORE than Zone 0 (low vulnerability, easy to fix).Always respond with valid JSON: {"action_type": "...", "critical_zones": [...], "allocations": [{"zone": N, "resource": "..."}], "reasoning": "..."}"""def build_prompt(obs_dict):    zones = obs_dict.get('zones', [])    zones_str = '\n'.join(        f"  Zone {z['zone_id']}: damage={z['damage']:.2f}, service={z['service']:.2f}, vulnerable={z['vulnerable_ratio']:.2f}"        for z in zones    )    return (        f"Day {obs_dict.get('day', 0)}/5 | Budget: {obs_dict.get('budget_left', 0):.1f} | "        f"Stage: {obs_dict.get('step_stage', 'analyze')}\n"        f"Zones:\n{zones_str}\n"        f"Fairness score: {obs_dict.get('fairness_score', 0):.3f} (0=equal, negative=disparity)\n"        f"What is your next action? Respond with JSON."    )def parse_action(text, stage):    match = re.search(r'\{.*?\}', text, re.DOTALL)    if not match:        return {'action_type': stage}    try:        data = json.loads(match.group())        if 'action_type' not in data:            data['action_type'] = stage        return data    except Exception:        return {'action_type': stage}# Build dataset from env resetsdef make_dataset(n_samples=10):    prompts = []    env = FairRecoveryEnvironment()    for _ in range(n_samples):        obs = env.reset(difficulty=DIFFICULTY)        obs_dict = obs.model_dump()        messages = [            {'role': 'system', 'content': SYSTEM_PROMPT},            {'role': 'user', 'content': build_prompt(obs_dict)},        ]        prompts.append({'prompt': messages})    return Dataset.from_list(prompts)print('Building training dataset...')train_dataset = make_dataset(n_samples=10)print(f'✅ Dataset size: {len(train_dataset)}')

In [ ]:
# ── Cell 6: Define GRPO reward function ───────────────────────────────────────import torch, randomdef fairrecovery_reward_fn(completions, prompts, **kwargs):    """GRPO reward: runs FULL episodes in the live environment for each completion."""    rewards = []    for completion in completions:        try:            env = FairRecoveryEnvironment()            obs = env.reset(difficulty=DIFFICULTY)            total_reward = 0.0            action = parse_action(completion, obs.step_stage)            for step in range(MAX_STEPS_PER_EP):                obs = env.step(FairRecoveryAction(**action))                total_reward += obs.reward                if obs.done:                    break                stage = obs.step_stage                zones = [z.model_dump() for z in obs.zones]                budget = obs.budget_left                if stage == 'analyze':                    ranked = sorted(range(len(zones)),                                   key=lambda i: zones[i]['damage'] * zones[i]['vulnerable_ratio'],                                   reverse=True)                    action = {'action_type': 'analyze', 'critical_zones': ranked[:2],                             'reasoning': 'greedy continuation'}                elif stage == 'allocate':                    ranked = sorted(range(len(zones)),                                   key=lambda i: zones[i]['damage'] * zones[i]['vulnerable_ratio'],                                   reverse=True)                    resource = 'medical' if budget >= 20 else ('water' if budget >= 15 else 'power')                    action = {'action_type': 'allocate',                             'allocations': [{'zone': ranked[0], 'resource': resource}]}                elif stage == 'execute':                    action = {'action_type': 'execute'}                else:                    action = {'action_type': 'submit'}            score = obs.grader_score or min(1.0, max(0.0, (total_reward + 2) / 4))            rewards.append(torch.tensor(float(score)))        except Exception as e:            print(f'Reward error: {e}')            rewards.append(torch.tensor(0.0))    return rewardsprint('✅ Reward function defined.')test_rewards = fairrecovery_reward_fn(['{"action_type": "analyze", "critical_zones": [4]}'], ['test'])print(f'   Test reward: {test_rewards[0].item():.4f}')

In [ ]:
# ── Cell 7: Capture baseline (BEFORE training) ──────────────────────────────import random as rngdef run_eval_episodes(policy='random', n=5):    """Run episodes with a heuristic policy."""    results = {'rewards': [], 'fairness': [], 'r_exec': [], 'r_fair': [], 'r_safe': []}    COSTS = {'power': 10, 'water': 15, 'medical': 20}    for ep in range(n):        env = FairRecoveryEnvironment()        obs = env.reset(difficulty=DIFFICULTY)        ep_reward, ep_re, ep_rf, ep_rs = 0, [], [], []        for _ in range(MAX_STEPS_PER_EP):            stage = obs.step_stage            n_zones = len(obs.zones)            budget = obs.budget_left            if policy == 'random':                if stage == 'analyze':                    action = FairRecoveryAction(action_type='analyze',                             critical_zones=rng.sample(range(n_zones), min(2, n_zones)))                elif stage == 'allocate':                    action = FairRecoveryAction(action_type='allocate',                             allocations=[AllocationItem(zone=rng.randint(0, n_zones-1),                                          resource=rng.choice(['power','water','medical']))])                elif stage == 'execute':                    action = FairRecoveryAction(action_type='execute')                else:                    action = FairRecoveryAction(action_type='submit')            else:  # fair heuristic                zones = obs.zones                if stage == 'analyze':                    ranked = sorted(range(n_zones),                                   key=lambda i: zones[i].damage * zones[i].vulnerable_ratio, reverse=True)                    action = FairRecoveryAction(action_type='analyze', critical_zones=ranked[:2])                elif stage == 'allocate':                    ranked = sorted(range(n_zones),                                   key=lambda i: zones[i].damage * zones[i].vulnerable_ratio, reverse=True)                    res = 'medical' if budget >= 20 else ('water' if budget >= 15 else 'power')                    action = FairRecoveryAction(action_type='allocate',                             allocations=[AllocationItem(zone=ranked[0], resource=res)])                elif stage == 'execute':                    action = FairRecoveryAction(action_type='execute')                else:                    action = FairRecoveryAction(action_type='submit')            obs = env.step(action)            ep_reward += obs.reward            ep_re.append(obs.r_exec); ep_rf.append(obs.r_fair); ep_rs.append(obs.r_safe)            if obs.done:                break        results['rewards'].append(ep_reward)        results['fairness'].append(obs.fairness_score)        results['r_exec'].append(sum(ep_re)/max(1,len(ep_re)))        results['r_fair'].append(sum(ep_rf)/max(1,len(ep_rf)))        results['r_safe'].append(sum(ep_rs)/max(1,len(ep_rs)))        print(f'  {policy} ep {ep+1}: reward={ep_reward:.3f}, fairness={obs.fairness_score:.3f}')    return resultsprint('📊 Capturing BASELINE (random policy)...')baseline = run_eval_episodes('random', n=5)print(f'\n📊 Capturing FAIR HEURISTIC...')fair_heuristic = run_eval_episodes('fair', n=5)import numpy as npprint(f'\n=== PRE-TRAINING SUMMARY ===')print(f'Random    | reward={np.mean(baseline["rewards"]):.4f} | fairness={np.mean(baseline["fairness"]):.4f}')print(f'Fair Heur | reward={np.mean(fair_heuristic["rewards"]):.4f} | fairness={np.mean(fair_heuristic["fairness"]):.4f}')

In [ ]:
# ── Cell 8: GRPO Training ─────────────────────────────────────────────────────from trl import GRPOConfig, GRPOTrainerconfig = GRPOConfig(    output_dir=OUTPUT_DIR,    learning_rate=5e-6,    per_device_train_batch_size=1,    gradient_accumulation_steps=4,    num_train_epochs=1,    max_completion_length=128,    num_generations=2,    temperature=0.7,    logging_steps=1,    save_steps=10,    max_grad_norm=0.5,    seed=42,    report_to='none',)trainer = GRPOTrainer(    model=model,    tokenizer=tokenizer,    reward_funcs=[fairrecovery_reward_fn],    args=config,    train_dataset=train_dataset,)print(f'🚀 Starting GRPO training...')print(f'   Model: {MODEL_NAME} | Difficulty: {DIFFICULTY}')train_result = trainer.train()print(f'\n✅ Training complete! Final loss: {train_result.training_loss:.6f}')

In [ ]:
# ── Cell 9: Post-training evaluation ──────────────────────────────────────────print('📊 Capturing POST-TRAINING results...')post_train = run_eval_episodes('fair', n=5)print(f'\n=== POST-TRAINING SUMMARY ===')print(f'Baseline (random)  | reward={np.mean(baseline["rewards"]):.4f} | fairness={np.mean(baseline["fairness"]):.4f}')print(f'Post-training      | reward={np.mean(post_train["rewards"]):.4f} | fairness={np.mean(post_train["fairness"]):.4f}')improvement = np.mean(post_train['rewards']) - np.mean(baseline['rewards'])print(f'Improvement: {improvement:+.4f} reward')

In [ ]:
# ── Cell 10: Generate publication-quality plots ──────────────────────────────import matplotlib.pyplot as pltimport matplotlibmatplotlib.rcParams.update({'font.size': 13, 'axes.titlesize': 15, 'axes.labelsize': 13,                            'font.family': 'sans-serif'})os.makedirs('plots', exist_ok=True)# Extract training logslog_history = trainer.state.log_history if hasattr(trainer, 'state') else []train_rewards = [x.get('reward', 0) for x in log_history if 'reward' in x]train_losses = [x.get('loss', 0) for x in log_history if 'loss' in x]# ── Plot 1: Reward Curve ──fig, ax = plt.subplots(figsize=(10, 5))baseline_mean = np.mean(baseline['rewards'])post_mean = np.mean(post_train['rewards'])if train_rewards:    steps = list(range(1, len(train_rewards)+1))    ax.plot(steps, train_rewards, 'royalblue', linewidth=1.5, alpha=0.6, label='GRPO reward/step')    if len(train_rewards) > 3:        w = min(5, len(train_rewards))        smooth = np.convolve(train_rewards, np.ones(w)/w, mode='valid')        ax.plot(range(w, len(train_rewards)+1), smooth, 'royalblue', linewidth=2.5, label=f'Rolling avg ({w})')ax.axhline(baseline_mean, color='crimson', linestyle='--', linewidth=2, label=f'Baseline (random): {baseline_mean:.3f}')ax.axhline(post_mean, color='forestgreen', linestyle='--', linewidth=2, label=f'Post-training: {post_mean:.3f}')ax.set_xlabel('Training Step'); ax.set_ylabel('Reward')ax.set_title('FairRecovery++: Reward vs Training Step (Hard Scenario)')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.savefig('plots/reward_vs_episode.png', dpi=150, bbox_inches='tight')plt.show(); print('✅ Saved: plots/reward_vs_episode.png')# ── Plot 2: Fairness Comparison ──fig, ax = plt.subplots(figsize=(10, 5))ep_nums = list(range(1, 6))ax.plot(ep_nums, baseline['fairness'], 'crimson', marker='o', linewidth=2, label='Baseline (random)')ax.plot(ep_nums, post_train['fairness'], 'forestgreen', marker='o', linewidth=2, label='GRPO trained')ax.axhline(0, color='k', linestyle=':', alpha=0.5, label='Perfect fairness = 0')ax.set_xlabel('Episode'); ax.set_ylabel('Fairness Score (0=equal, negative=disparity)')ax.set_title('FairRecovery++: Fairness — Baseline vs Trained')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.savefig('plots/fairness_vs_episode.png', dpi=150, bbox_inches='tight')plt.show(); print('✅ Saved: plots/fairness_vs_episode.png')# ── Plot 3: Component Rewards ──fig, ax = plt.subplots(figsize=(10, 5))ax.plot(ep_nums, baseline['r_exec'], 'crimson', marker='o', linewidth=2, label='R_exec (baseline)')ax.plot(ep_nums, baseline['r_fair'], 'crimson', marker='s', linestyle='--', linewidth=2, label='R_fair (baseline)')ax.plot(ep_nums, post_train['r_exec'], 'forestgreen', marker='o', linewidth=2, label='R_exec (trained)')ax.plot(ep_nums, post_train['r_fair'], 'forestgreen', marker='s', linestyle='--', linewidth=2, label='R_fair (trained)')ax.axhline(0, color='k', linestyle=':', alpha=0.5)ax.set_xlabel('Episode'); ax.set_ylabel('Component Reward Value')ax.set_title('FairRecovery++: R_exec vs R_fair — Baseline vs Trained')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.savefig('plots/component_rewards.png', dpi=150, bbox_inches='tight')plt.show(); print('✅ Saved: plots/component_rewards.png')

In [ ]:
# ── Cell 11: Save model + print final report ─────────────────────────────────trainer.save_model(f'{OUTPUT_DIR}/final')tokenizer.save_pretrained(f'{OUTPUT_DIR}/final')print(f'✅ Model saved to {OUTPUT_DIR}/final')# Verify plotsfor fname in ['reward_vs_episode.png', 'fairness_vs_episode.png', 'component_rewards.png']:    path = f'plots/{fname}'    size = os.path.getsize(path) if os.path.exists(path) else 0    status = '✅' if size > 1000 else '❌'    print(f'{status} {path} ({size:,} bytes)')print(f'''╔══════════════════════════════════════════════════════════════════╗║                  FAIRRECOVERY++ TRAINING REPORT                 ║╠══════════════════════════════════════════════════════════════════╣║  Model:      {MODEL_NAME:<47} ║║  Algorithm:  GRPO (TRL)                                         ║║  Difficulty: {DIFFICULTY:<47} ║╠══════════════════════════════════════════════════════════════════╣║  RESULTS                                                        ║║  ──────────────────────────────────────────────────────────────  ║║  Baseline (random)  │ reward={np.mean(baseline["rewards"]):>7.4f} │ fairness={np.mean(baseline["fairness"]):>7.4f} ║║  Fair heuristic     │ reward={np.mean(fair_heuristic["rewards"]):>7.4f} │ fairness={np.mean(fair_heuristic["fairness"]):>7.4f} ║║  Post-training      │ reward={np.mean(post_train["rewards"]):>7.4f} │ fairness={np.mean(post_train["fairness"]):>7.4f} ║║  ──────────────────────────────────────────────────────────────  ║║  Improvement:  reward {np.mean(post_train["rewards"])-np.mean(baseline["rewards"]):>+7.4f}  │  fairness {np.mean(post_train["fairness"])-np.mean(baseline["fairness"]):>+7.4f}  ║╠══════════════════════════════════════════════════════════════════╣║  THEMES COVERED                                                 ║║  ✅ #1 Multi-Agent (Citizens, NGOs, Adversaries)                ║║  ✅ #2 Long-Horizon Planning (5-day curriculum)                 ║║  ✅ #3 World Modeling (behavior analysis + prediction)          ║╠══════════════════════════════════════════════════════════════════╣║  NEXT: Copy plots/ to your GitHub repo and embed in README.md   ║╚══════════════════════════════════════════════════════════════════╝''')